# F.E. Walter prompt comparison
Compare the Seasonal Target and DKP teaching configurations, inspect the exact prompt files each one selects, and validate both cases without contacting a model provider.

In [ ]:
from pathlib import Path
import json, subprocess, sys, yaml

root = Path.cwd()
if not (root / 'examples' / 'fe_walter' / 'seasonal_target.yaml').exists():
    root = root.parent
example_dir = root / 'examples' / 'fe_walter'
config_paths = {
    'Seasonal Target': example_dir / 'seasonal_target.yaml',
    'DKP': example_dir / 'dkp.yaml',
}
configs = {name: yaml.safe_load(path.read_text(encoding='utf-8')) for name, path in config_paths.items()}
case = json.loads((example_dir / 'case.json').read_text(encoding='utf-8'))
dates = [row['date'] for row in case['inflow_records']]
{'days': len(dates), 'start': dates[0], 'end': dates[-1], 'reservoirs': list(case['network']['reservoirs'])}

## Inspect the selected prompts
Both cases share the same synthetic inflow and physical state. The YAML files select different named prompt files, and only DKP attaches local operating guidance. These teaching prompts are not copies of historical experiment prompts.

In [ ]:
prompt_texts = {}
comparison = {}
for name, config in configs.items():
    agent = config['agents']['walter_operator']
    selected = {kind: example_dir / relative for kind, relative in agent['prompt'].items()}
    prompt_texts[name] = {kind: path.read_text(encoding='utf-8') for kind, path in selected.items()}
    comparison[name] = {
        'system_prompt': selected['system'].relative_to(root).as_posix(),
        'user_prompt': selected['user'].relative_to(root).as_posix(),
        'knowledge': agent.get('knowledge', {}),
        'model': config['models']['local']['model'],
        'base_url': config['models']['local']['base_url'],
    }
comparison

## Validate both configurations offline
`check` resolves the case, prompt, knowledge, and model settings. It does not contact Ollama and does not create run output.

In [ ]:
command = [sys.executable, '-m', 'llm_reservoir_agent.workflow', 'check']
check_results = {}
for name, path in config_paths.items():
    completed = subprocess.run(command + [str(path)], text=True, capture_output=True)
    if completed.returncode:
        raise RuntimeError(completed.stderr or completed.stdout)
    check_results[name] = json.loads(completed.stdout)
check_summary = {
    name: {
        'schema_version': result['schema_version'],
        'days': result['days'],
        'network_calls': result['network_calls'],
        'live_ready': result['live_ready'],
    }
    for name, result in check_results.items()
}
check_summary

## Optional local Ollama runs
The next cell prints commands only. Review the prompts first, start Ollama with `gemma4:12b`, choose fresh `run_id` and `output` values, and then run a command manually. `--allow-network` explicitly authorizes requests to the configured local endpoint.

In [ ]:
opt_in_commands = [
    'reservoir-experiment run examples/fe_walter/seasonal_target.yaml --allow-network',
    'reservoir-experiment run examples/fe_walter/dkp.yaml --allow-network',
]
print('\n'.join(opt_in_commands))

## Next steps
Read the [F.E. Walter guide](../examples/fe_walter/README.md) for verification and output paths. Use the [results guide](../docs/RESULTS_GUIDE.md) to interpret decisions, provider attempts, physical trajectories, and provenance.